# Verificación de funcionamiento de normalización

La normalización funciona dado que, para todas las variables cuantitativas, la media se aproxima a 0 y la desviación estándar equivale a 1.

## Preparación del dataset
Se realiza la inserción de la ruta para utilizar archivos del proyecto, y se carga el dataset normalizado.

In [5]:
import polars as pl
import sys
from pathlib import Path

ROOT = Path.cwd().parent 
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from config.rutas import RUTA_DATA_PROCESSED
lectura_normalizada = pl.read_csv(RUTA_DATA_PROCESSED / "normalized_consumption.csv")

## Generación e impresión de tabla con medias y desviaciones

### Función de tabla de medidas

In [ ]:
# Proporción de entrenamiento
def media_y_std(df:pl.Dataframe, prop_entrenamiento:float=0.8) -> pl.Dataframe:

    # Columnas numéricas
    cols_num = [
            col
            for col, dtype
            in df.schema.items()
            if dtype.is_numeric() and col != "datetime"
        ]

    # Punto de corte para entrenamiento (80%)
    corte_entrenamiento = int(df.height * prop_entrenamiento)
    df_entr = df.slice(0, corte_entrenamiento)
    df_test = df.slice(corte_entrenamiento)

    # Media y std
    expresiones = []
    for col in cols_num:
        expresiones.extend([
            pl.col(col).mean().round(6).alias(f"media_{col}"),
            pl.col(col).std().round(6).alias(f"std_{col}")
        ])

    # Resúmenes
    resumen_entr = df_entr.select(expresiones).with_columns(pl.lit("Entrenamiento (80%)").alias("particion"))
    resumen_test = df_test.select(expresiones).with_columns(pl.lit("Prueba (20%)").alias("particion"))

    results = pl.concat([resumen_entr, resumen_test]).select(
        ["particion"] + [col for col in resumen_entr.columns if col != "particion"]
    )
    return results

### Cálculo para dataset normalizado

In [9]:
pl.Config.set_tbl_cols(-1)        
pl.Config.set_tbl_rows(-1)        
pl.Config.set_tbl_width_chars(1000)
print(media_y_std(lectura_normalizada))

NameError: name 'prop_entr' is not defined